In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
transfer_learning_df = pd.read_csv("final_accuracies/transfer_learning/dinov2base_baseline_and_skips.csv")

In [16]:
for dataset in transfer_learning_df["dataset"].unique():
    dataset_df = transfer_learning_df[transfer_learning_df["dataset"] == dataset]
    baseline_acc = dataset_df[dataset_df["approx_layer"] == "[]"].iloc[0]["accuracy_mean"]
    #dataset_df["H_s"] = dataset_df.apply(lambda x: int(x["approx_layer"].split("(")[1].split(",")[0]) if x["approx_layer"] != "[]" else -1, axis=1)

In [17]:
dataset_df.head(10)

,model,dataset,approx_layer,translator,mlp_linearize,mlp_mode,attn_linearize,attn_mode,head_dict,fit_dataset,accuracy_mean,accuracy_std,orig_head_accuracy_mean,orig_head_accuracy_std,order
3,facebook/dinov2-base,chestxray14,[],linear,[],identity,[],identity,{},chestxray14,0.594185,0.078908,NaN,NaN,9
7,facebook/dinov2-base,chestxray14,"[(0, 1)]",linear,[],identity,[],identity,{},cifar100,0.602361,0.064974,NaN,NaN,9
11,facebook/dinov2-base,chestxray14,"[(2, 3)]",linear,[],identity,[],identity,{},cifar100,0.641812,0.038087,NaN,NaN,9
15,facebook/dinov2-base,chestxray14,"[(4, 5)]",linear,[],identity,[],identity,{},cifar100,0.641010,0.043699,NaN,NaN,9
19,facebook/dinov2-base,chestxray14,"[(6, 7)]",linear,[],identity,[],identity,{},cifar100,0.575455,0.055406,NaN,NaN,9
23,facebook/dinov2-base,chestxray14,"[(8, 9)]",linear,[],identity,[],identity,{},cifar100,0.591925,0.088742,NaN,NaN,9
27,facebook/dinov2-base,chestxray14,"[(10, 11)]",linear,[],identity,[],identity,{},cifar100,0.593174,0.094666,NaN,NaN,9


In [ ]:
# rad-dino / rad-dino-maira-2 x chestxray14 / chestxray_pneumonia: original (own-fit) vs
# cifar100-donor transfer vs the *other* chest dataset as donor (in-domain cross-fit).
group1_own = pd.concat([
    pd.read_csv("final_accuracies/group1_chestxray14.csv"),
    pd.read_csv("final_accuracies/group1_chestxray_pneumonia.csv"),
], ignore_index=True)
group1_own = group1_own[group1_own["fit_dataset"] == group1_own["dataset"]]

cifar100_reuse = pd.read_csv("final_accuracies/transfer_learning/translator_reuse_cifar100.csv")
cifar100_reuse = cifar100_reuse[cifar100_reuse["fit_dataset"] == "cifar100"]

cross_fit = pd.concat([
    pd.read_csv("final_accuracies/transfer_learning/group1_cross_fit_chestxray_pneumonia.csv"),
    pd.read_csv("final_accuracies/transfer_learning/group1_cross_fit_chestxray14.csv"),
], ignore_index=True)
cross_fit = cross_fit[cross_fit["fit_dataset"] != cross_fit["dataset"]]


def add_hs(df):
    # H_s: skip start layer index (-1 for the no-skip baseline). skip_length: number of blocks
    # skipped (0 for baseline) -- own-fit baseline files cover skip lengths 1-4, but the
    # translator-reuse/cross-fit sweeps only ever cover length-1 spans, so own-fit needs the
    # same length==1 restriction to be a fair like-for-like comparison on the same x-axis.
    df = df.copy()
    df["H_s"] = df["approx_layer"].apply(
        lambda x: int(x.split("(")[1].split(",")[0]) if x != "[]" else -1
    )
    df["skip_length"] = df["approx_layer"].apply(
        lambda x: int(x.split(",")[1].split(")")[0]) - int(x.split("(")[1].split(",")[0]) if x != "[]" else 0
    )
    return df


group1_own = add_hs(group1_own)
cifar100_reuse = add_hs(cifar100_reuse)
cross_fit = add_hs(cross_fit)

In [ ]:
combos = [
    ("microsoft/rad-dino", "chestxray14"),
    ("microsoft/rad-dino", "chestxray_pneumonia"),
    ("microsoft/rad-dino-maira-2", "chestxray14"),
    ("microsoft/rad-dino-maira-2", "chestxray_pneumonia"),
]

for model, dataset in combos:
    own_df = group1_own[
        (group1_own["model"] == model) & (group1_own["dataset"] == dataset)
        & (group1_own["skip_length"] == 1)
    ].sort_values("H_s")
    baseline_acc = group1_own[
        (group1_own["model"] == model) & (group1_own["dataset"] == dataset) & (group1_own["H_s"] == -1)
    ].iloc[0]["accuracy_mean"]

    cifar_df = cifar100_reuse[
        (cifar100_reuse["model"] == model) & (cifar100_reuse["dataset"] == dataset)
        & (cifar100_reuse["skip_length"] == 1)
    ].sort_values("H_s")

    other_donor = "chestxray_pneumonia" if dataset == "chestxray14" else "chestxray14"
    other_df = cross_fit[
        (cross_fit["model"] == model) & (cross_fit["dataset"] == dataset)
        & (cross_fit["fit_dataset"] == other_donor) & (cross_fit["skip_length"] == 1)
    ].sort_values("H_s")

    plt.figure(figsize=(10, 6))
    plt.plot(own_df["H_s"], own_df["accuracy_mean"], marker='o', label='Original (own-fit) Accuracy')
    plt.fill_between(own_df["H_s"], own_df["accuracy_mean"] - own_df["accuracy_std"],
                      own_df["accuracy_mean"] + own_df["accuracy_std"], alpha=0.2)

    plt.plot(cifar_df["H_s"], cifar_df["accuracy_mean"], marker='o', label='CIFAR-100 Transfer Accuracy')
    plt.fill_between(cifar_df["H_s"], cifar_df["accuracy_mean"] - cifar_df["accuracy_std"],
                      cifar_df["accuracy_mean"] + cifar_df["accuracy_std"], alpha=0.2)

    plt.plot(other_df["H_s"], other_df["accuracy_mean"], marker='o', label=f'{other_donor} Transfer Accuracy')
    plt.fill_between(other_df["H_s"], other_df["accuracy_mean"] - other_df["accuracy_std"],
                      other_df["accuracy_mean"] + other_df["accuracy_std"], alpha=0.2)

    plt.axhline(y=baseline_acc, color='r', linestyle='--', label=f'Baseline: {baseline_acc:.2f}')
    plt.title(f'Accuracy Comparison for {model} on {dataset}')
    plt.xlabel('H_s (Layer Index)')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
# Same structure for group2: vit-large-patch16-224 / phikon-v2 x patchcamelyon / tcga_ut.
# cifar100 donor is the out-of-domain transfer; the other group2 domain-shifted dataset is the
# in-domain donor (patchcamelyon <-> tcga_ut), same pairing logic as group1's two chest datasets.
group2_own = pd.concat([
    pd.read_csv("final_accuracies/group2_patchcamelyon.csv"),
    pd.read_csv("final_accuracies/group2_tcga_ut.csv"),
], ignore_index=True)
group2_own = group2_own[group2_own["fit_dataset"] == group2_own["dataset"]]

group2_cifar100_reuse = pd.read_csv("final_accuracies/transfer_learning/translator_reuse_cifar100.csv")
group2_cifar100_reuse = group2_cifar100_reuse[group2_cifar100_reuse["fit_dataset"] == "cifar100"]

group2_cross_fit = pd.concat([
    pd.read_csv("final_accuracies/transfer_learning/group2_cross_fit_patchcamelyon.csv"),
    pd.read_csv("final_accuracies/transfer_learning/group2_cross_fit_tcga_ut.csv"),
], ignore_index=True)
group2_cross_fit = group2_cross_fit[group2_cross_fit["fit_dataset"] != group2_cross_fit["dataset"]]

# own-fit's own "skip_length" column (from group2_combined) is dropped in favor of the same
# add_hs-derived one used everywhere else, so the length==1 filter means the same thing
# consistently across own/cifar100/cross-fit regardless of which file it came from.
group2_own = add_hs(group2_own.drop(columns=["skip_length"]))
group2_cifar100_reuse = add_hs(group2_cifar100_reuse)
group2_cross_fit = add_hs(group2_cross_fit)

In [ ]:
group2_combos = [
    ("google/vit-large-patch16-224", "patchcamelyon"),
    ("google/vit-large-patch16-224", "tcga_ut"),
    ("owkin/phikon-v2", "patchcamelyon"),
    ("owkin/phikon-v2", "tcga_ut"),
]

for model, dataset in group2_combos:
    own_df = group2_own[
        (group2_own["model"] == model) & (group2_own["dataset"] == dataset)
        & (group2_own["skip_length"] == 1)
    ].sort_values("H_s")
    baseline_acc = group2_own[
        (group2_own["model"] == model) & (group2_own["dataset"] == dataset) & (group2_own["H_s"] == -1)
    ].iloc[0]["accuracy_mean"]

    cifar_df = group2_cifar100_reuse[
        (group2_cifar100_reuse["model"] == model) & (group2_cifar100_reuse["dataset"] == dataset)
        & (group2_cifar100_reuse["skip_length"] == 1)
    ].sort_values("H_s")

    other_donor = "tcga_ut" if dataset == "patchcamelyon" else "patchcamelyon"
    other_df = group2_cross_fit[
        (group2_cross_fit["model"] == model) & (group2_cross_fit["dataset"] == dataset)
        & (group2_cross_fit["fit_dataset"] == other_donor) & (group2_cross_fit["skip_length"] == 1)
    ].sort_values("H_s")

    plt.figure(figsize=(10, 6))
    plt.plot(own_df["H_s"], own_df["accuracy_mean"], marker='o', label='Original (own-fit) Accuracy')
    plt.fill_between(own_df["H_s"], own_df["accuracy_mean"] - own_df["accuracy_std"],
                      own_df["accuracy_mean"] + own_df["accuracy_std"], alpha=0.2)

    plt.plot(cifar_df["H_s"], cifar_df["accuracy_mean"], marker='o', label='CIFAR-100 Transfer Accuracy')
    plt.fill_between(cifar_df["H_s"], cifar_df["accuracy_mean"] - cifar_df["accuracy_std"],
                      cifar_df["accuracy_mean"] + cifar_df["accuracy_std"], alpha=0.2)

    plt.plot(other_df["H_s"], other_df["accuracy_mean"], marker='o', label=f'{other_donor} Transfer Accuracy')
    plt.fill_between(other_df["H_s"], other_df["accuracy_mean"] - other_df["accuracy_std"],
                      other_df["accuracy_mean"] + other_df["accuracy_std"], alpha=0.2)

    plt.axhline(y=baseline_acc, color='r', linestyle='--', label=f'Baseline: {baseline_acc:.2f}')
    plt.title(f'Accuracy Comparison for {model} on {dataset}')
    plt.xlabel('H_s (Layer Index)')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
import os

REPORT_FIG_DIR = "report_figures"
os.makedirs(REPORT_FIG_DIR, exist_ok=True)


def plot_combo_on_ax(ax, own_df, cifar_df, other_df, other_donor, baseline_acc, model, dataset):
    ax.plot(own_df["H_s"], own_df["accuracy_mean"], marker='o', label='Original (own-fit) Accuracy')
    ax.fill_between(own_df["H_s"], own_df["accuracy_mean"] - own_df["accuracy_std"],
                     own_df["accuracy_mean"] + own_df["accuracy_std"], alpha=0.2)

    ax.plot(cifar_df["H_s"], cifar_df["accuracy_mean"], marker='o', label='CIFAR-100 Transfer Accuracy')
    ax.fill_between(cifar_df["H_s"], cifar_df["accuracy_mean"] - cifar_df["accuracy_std"],
                     cifar_df["accuracy_mean"] + cifar_df["accuracy_std"], alpha=0.2)

    ax.plot(other_df["H_s"], other_df["accuracy_mean"], marker='o', label=f'{other_donor} Transfer Accuracy')
    ax.fill_between(other_df["H_s"], other_df["accuracy_mean"] - other_df["accuracy_std"],
                     other_df["accuracy_mean"] + other_df["accuracy_std"], alpha=0.2)

    ax.axhline(y=baseline_acc, color='r', linestyle='--', label=f'Baseline: {baseline_acc:.2f}')
    ax.set_title(f'{model} on {dataset}')
    ax.set_xlabel('H_s (Layer Index)')
    ax.set_ylabel('Accuracy')
    # Every subplot gets its own legend rather than sharing one: the green line's label
    # (the "other" donor dataset) changes per panel, so a single shared legend would
    # mislabel it everywhere except the panel it was drawn from.
    ax.legend(fontsize=8)


def plot_grid(own, cifar, cross, combos, other_donor_fn, suptitle, save_path):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    for ax, (model, dataset) in zip(axes.flat, combos):
        own_df = own[(own["model"] == model) & (own["dataset"] == dataset)
                      & (own["skip_length"] == 1)].sort_values("H_s")
        baseline_acc = own[(own["model"] == model) & (own["dataset"] == dataset)
                            & (own["H_s"] == -1)].iloc[0]["accuracy_mean"]
        cifar_df = cifar[(cifar["model"] == model) & (cifar["dataset"] == dataset)
                          & (cifar["skip_length"] == 1)].sort_values("H_s")
        other_donor = other_donor_fn(dataset)
        other_df = cross[(cross["model"] == model) & (cross["dataset"] == dataset)
                          & (cross["fit_dataset"] == other_donor) & (cross["skip_length"] == 1)].sort_values("H_s")
        plot_combo_on_ax(ax, own_df, cifar_df, other_df, other_donor, baseline_acc, model, dataset)
    fig.suptitle(suptitle, fontsize=14)
    fig.tight_layout(rect=[0, 0, 1, 0.96])
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"saved {save_path}")

In [ ]:
plot_grid(
    group1_own, cifar100_reuse, cross_fit, combos,
    other_donor_fn=lambda dataset: "chestxray_pneumonia" if dataset == "chestxray14" else "chestxray14",
    suptitle="Group 1 (rad-dino / rad-dino-maira-2): translator transfer accuracy",
    save_path=os.path.join(REPORT_FIG_DIR, "group1_translator_reuse_grid.png"),
)

In [ ]:
plot_grid(
    group2_own, group2_cifar100_reuse, group2_cross_fit, group2_combos,
    other_donor_fn=lambda dataset: "tcga_ut" if dataset == "patchcamelyon" else "patchcamelyon",
    suptitle="Group 2 (vit-large-patch16-224 / phikon-v2): translator transfer accuracy",
    save_path=os.path.join(REPORT_FIG_DIR, "group2_translator_reuse_grid.png"),
)